# Extra · Redes neurais por dentro
**Aprendizagem Computacional · 4º TIA · Apostila do Aluno · notebook extra · versão INICIAL**

Complemento do **cap. 15** da apostila. Em vez de usar a rede como caixa-preta, vamos construí-la:

| Seção | O que você vai ver | Apostila |
|---|---|---|
| 1 | O neurônio: soma ponderada + ativação | cap. 15 |
| 2 | Perceptron do zero: aprende o E lógico, falha no OU-exclusivo (XOR) | cap. 15 e 10 (dimensão VC) |
| 3 | Rede com uma camada escondida, do zero, com retropropagação: resolve o XOR | cap. 15 |
| 4 | MLP do scikit-learn em imagens de dígitos manuscritos, contra baseline e logística | cap. 9, 12, 15 |
| 5 | Sobreajuste e regularização na rede | cap. 10 |

Tudo roda sem internet e em menos de 1 minuto. Próximo passo: o notebook extra de **visão computacional**.

**Como usar a versão INICIAL:** há 3 lacunas `# TODO`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
SEMENTE = 42
rng = np.random.default_rng(SEMENTE)

## 1 · O neurônio artificial
Um neurônio faz duas coisas: (1) uma **soma ponderada** das entradas mais um viés, z = w·x + b; (2) passa z por uma **função de ativação** f(z). Sem a ativação não linear, empilhar camadas não adianta: uma combinação de retas continua sendo uma reta.

In [ ]:
z = np.linspace(-6, 6, 200)
ativacoes = {
    'degrau (perceptron)': (z > 0).astype(float),
    'sigmoide (logística)': 1 / (1 + np.exp(-z)),
    'ReLU (redes modernas)': np.maximum(0, z),
}
fig, eixos = plt.subplots(1, 3, figsize=(12, 3))
for ax, (nome, f) in zip(eixos, ativacoes.items()):
    ax.plot(z, f, lw=2); ax.set_title(nome); ax.axhline(0, color='grey', lw=0.5); ax.axvline(0, color='grey', lw=0.5)
plt.tight_layout(); plt.show()

## 2 · Perceptron do zero
O perceptron (Rosenblatt, 1958) aprende assim: para cada exemplo, se errou, empurra os pesos na direção certa:

`w ← w + taxa × (y − previsto) × x`

Vamos ensinar dois problemas com duas entradas 0/1: o **E lógico** (só 1 se ambas forem 1) e o **OU-exclusivo, XOR** (1 se forem diferentes).

In [ ]:
X_logico = np.array([[0, 0], [0, 1], [1, 0], [1, 1]])
y_E = np.array([0, 0, 0, 1])
y_XOR = np.array([0, 1, 1, 0])

def treinar_perceptron(X, y, epocas=20, taxa=0.1):
    w, b = np.zeros(X.shape[1]), 0.0
    for _ in range(epocas):
        for xi, yi in zip(X, y):
            previsto = int(xi @ w + b > 0)
            erro = yi - previsto
            # TODO: regra do perceptron. Atualize w e b usando taxa, erro e xi
            w = ...
            b = ...
    return w, b

for nome, y_alvo in [('E lógico', y_E), ('XOR', y_XOR)]:
    w, b = treinar_perceptron(X_logico, y_alvo)
    previsto = (X_logico @ w + b > 0).astype(int)
    print(f'{nome:9s} alvo={y_alvo} previsto={previsto} acertos={int((previsto == y_alvo).sum())}/4')

**Leitura:** o perceptron acerta 4/4 no E lógico, mas nunca passa de 3/4 no XOR, por mais épocas que você dê. Desenhe os 4 pontos num papel: nenhuma reta separa os dois 1 dos dois 0 no XOR. Isso é a **dimensão VC** do cap. 10 em ação: um classificador linear não tem capacidade para essa rotulagem.

## 3 · Uma camada escondida resolve: retropropagação do zero
Rede com 2 entradas → 4 neurônios escondidos (ReLU) → 1 saída (sigmoide). O treino repete:
1. **ida (forward):** calcula a previsão;
2. **erro:** compara com o alvo (entropia cruzada);
3. **volta (backpropagation):** calcula quanto cada peso contribuiu para o erro;
4. **descida do gradiente:** ajusta cada peso um pouco na direção que reduz o erro.

In [ ]:
def relu(z):
    return ...  # TODO: ReLU devolve z quando z > 0 e 0 caso contrário (dica: np.maximum)

W1 = rng.normal(0, 1, (2, 4)); b1 = np.zeros(4)
W2 = rng.normal(0, 1, (4, 1)); b2 = np.zeros(1)
y_col = y_XOR.reshape(-1, 1)
taxa, historico = 0.1, []

for epoca in range(3000):
    # ida
    z1 = X_logico @ W1 + b1; a1 = relu(z1)
    z2 = a1 @ W2 + b2;       saida = 1 / (1 + np.exp(-z2))
    perda = -np.mean(y_col * np.log(saida + 1e-9) + (1 - y_col) * np.log(1 - saida + 1e-9))
    historico.append(perda)
    # volta (regra da cadeia)
    d2 = (saida - y_col) / len(X_logico)
    dW2 = a1.T @ d2;          db2 = d2.sum(axis=0)
    d1 = (d2 @ W2.T) * (z1 > 0)
    dW1 = X_logico.T @ d1;    db1 = d1.sum(axis=0)
    # descida do gradiente
    W1 -= taxa * dW1; b1 -= taxa * db1; W2 -= taxa * dW2; b2 -= taxa * db2

print('Probabilidades finais:', saida.ravel().round(3), '| previsto:', (saida.ravel() > 0.5).astype(int), '| alvo:', y_XOR)
plt.plot(historico); plt.xlabel('época'); plt.ylabel('perda (entropia cruzada)'); plt.title('A rede aprendendo o XOR'); plt.show()

**Pergunta para discussão:** o que mudou em relação ao perceptron para o XOR passar a ser resolvido?

**Resposta esperada:** a camada escondida com ativação não linear. Os 4 neurônios escondidos criam novas \"colunas\" a partir das entradas, e nesse novo espaço os pontos ficam separáveis. É exatamente o que as redes fazem em grande escala: aprendem representações.

## 4 · MLP em imagens: dígitos manuscritos
Agora um problema de verdade: 1.797 imagens de dígitos de 0 a 9, com **8 × 8 pixels** (64 colunas, tons de 0 a 16). Cada imagem vira uma linha de 64 números. É a ponte para o notebook de visão computacional.

In [ ]:
from sklearn.datasets import load_digits
digitos = load_digits()
X, y = digitos.data, digitos.target
fig, eixos = plt.subplots(1, 8, figsize=(10, 1.6))
for ax, img, rot in zip(eixos, digitos.images, y):
    ax.imshow(img, cmap='gray_r'); ax.set_title(rot); ax.axis('off')
plt.show()
print(X.shape, '| classes:', np.bincount(y))

In [ ]:
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEMENTE)
cv = StratifiedKFold(5, shuffle=True, random_state=SEMENTE)
modelos = {
    'Baseline (classe mais frequente)': DummyClassifier(strategy='most_frequent'),
    'Regressão logística': make_pipeline(StandardScaler(), LogisticRegression(max_iter=3000)),
    # TODO (cap. 15): pipeline com StandardScaler + MLPClassifier(hidden_layer_sizes=(64,), alpha=1e-3, max_iter=1000, random_state=SEMENTE)
    'MLP (64 neurônios)': None,
}
resultado = pd.DataFrame({n: cross_val_score(m, X_tr, y_tr, cv=cv) for n, m in modelos.items()}).agg(['mean', 'std']).T
resultado.round(3)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
mlp = modelos['MLP (64 neurônios)'].fit(X_tr, y_tr)
print(f'TESTE (uma vez) · acurácia do MLP: {mlp.score(X_te, y_te):.3f}')
ConfusionMatrixDisplay.from_estimator(mlp, X_te, y_te, cmap='Greens', colorbar=False)
plt.title('Onde o MLP confunde'); plt.show()
erros = np.where(mlp.predict(X_te) != y_te)[0][:8]
fig, eixos = plt.subplots(1, len(erros), figsize=(10, 1.8))
for ax, i in zip(np.atleast_1d(eixos), erros):
    ax.imshow(X_te[i].reshape(8, 8), cmap='gray_r'); ax.set_title(f'real {y_te[i]}\nprev {mlp.predict(X_te[i:i+1])[0]}', fontsize=9); ax.axis('off')
plt.show()

**Leitura:** o baseline acerta cerca de 10% (são 10 classes equilibradas). Logística e MLP passam de 95%. Olhar os erros é a análise de erros do cap. 12: muitos são dígitos mal escritos que até uma pessoa hesitaria em ler.

**Pergunta para discussão:** se a logística já vai tão bem, para que a rede?

**Resposta esperada:** nestas imagens minúsculas e centralizadas, um modelo linear já basta. Quando a imagem é grande, o objeto muda de posição e o fundo varia (uma placa numa câmera de rua), modelos lineares e MLPs falham, e entram as redes **convolucionais**: é o próximo notebook.

## 5 · Sobreajuste e regularização na rede (cap. 10)
Mesma rede, variando só a regularização `alpha` (penalidade L2 nos pesos). Pouca regularização: a rede decora o treino. Regularização demais: ela não consegue aprender.

In [ ]:
from sklearn.model_selection import cross_validate
alphas = [1e-5, 1e-3, 1e-1, 1, 10, 100]
linhas = []
for a in alphas:
    m = make_pipeline(StandardScaler(), MLPClassifier(hidden_layer_sizes=(64,), alpha=a, max_iter=1000, random_state=SEMENTE))
    r = cross_validate(m, X_tr, y_tr, cv=cv, return_train_score=True)
    linhas.append({'alpha': a, 'treino': r['train_score'].mean(), 'validação': r['test_score'].mean()})
curva = pd.DataFrame(linhas)
plt.semilogx(curva['alpha'], curva['treino'], marker='o', label='treino')
plt.semilogx(curva['alpha'], curva['validação'], marker='o', label='validação (CV)')
plt.xlabel('alpha (regularização)'); plt.ylabel('acurácia'); plt.legend(); plt.grid(alpha=0.3); plt.show()
curva.round(3)

**Leitura:** com alpha muito alto, treino e validação despencam juntos (subajuste, viés alto). Com alpha baixo, o treino fica perto de 100% e a validação um pouco abaixo (a folga do cap. 10). O melhor alpha é escolhido pela validação, nunca pelo teste.

## Checklist
- [ ] Sei explicar soma ponderada + ativação e por que a não linearidade é essencial.
- [ ] Entendi por que o perceptron falha no XOR e a camada escondida resolve.
- [ ] Sei descrever as 4 etapas do treino: ida, erro, volta, ajuste.
- [ ] Comparei a rede com baseline e logística no mesmo protocolo.
- [ ] Escolhi a regularização pela validação cruzada.